In [1]:
import os
os.getcwd()

'd:\\GitHub\\Sinusoidal-Distribution\\python'

In [2]:
%run sinu.ipynb

In [3]:
import torch
import torch.nn as nn
from torch.autograd import Function
import numpy as np
from scipy.integrate import quad

class SinuCDFFunction(Function):
    @staticmethod
    def forward(ctx, z, s, k):
        """
        z: Pre-activation tensor (Batch, Nodes)
        s: Shape parameter (Scalar)
        k: Concentration parameter (Scalar)
        """
        device = z.device
        z_flat = z.detach().cpu().numpy().flatten()
        s_val, k_val = s.item(), k.item()

        # Vectorized evaluation of your psinustd
        # We use a wrapper to handle the array input from numpy
        a_flat = psinustd(z_flat, s_val, k_val)
        
        a = torch.from_numpy(a_flat).to(device).view_as(z)
        ctx.save_for_backward(z, s, k)
        return a

    @staticmethod
    def backward(ctx, grad_output):
        z, s, k = ctx.saved_tensors
        device = z.device
        z_np = z.detach().cpu().numpy()
        s_val, k_val = s.item(), k.item()

        # 1. Gradient w.r.t Input z: dL/dz = dL/da * f(z)
        # The derivative of the CDF is the PDF (dsinustd)
        f_z_np = dsinustd(z_np.flatten(), s_val, k_val)
        f_z = torch.from_numpy(f_z_np).to(device).view_as(z)
        grad_z = grad_output * f_z

        # 2. Gradients w.r.t Shape Parameters s and k
        # These require the partial integrals of the sensitivity functions
        # We compute these via numerical quadrature over the batch
        grad_s = torch.zeros(1, device=device)
        grad_k = torch.zeros(1, device=device)

        # Sensitivity functions for the shape parameters
        def sens_s(t):
            # Partial derivative of the density w.r.t s
            # Derived via: d/ds [sin^k(pi*t^s) / A]
            # This is simplified for numerical stability
            eps = 1e-5
            return (psinustd(t, s_val + eps, k_val) - psinustd(t, s_val, k_val)) / eps

        def sens_k(t):
            eps = 1e-5
            return (psinustd(t, s_val, k_val + eps) - psinustd(t, s_val, k_val)) / eps

        # Summing contributions across the batch and nodes
        # In practice, for a common s,k, we use the mean gradient
        z_sample = z_np.mean() # Representative z for shape update
        if 0 < z_sample < 1:
            grad_s = torch.tensor([(grad_output * sens_s(z_sample)).sum()], device=device)
            grad_k = torch.tensor([(grad_output * sens_k(z_sample)).sum()], device=device)

        return grad_z, grad_s, grad_k

class SinuNetwork(nn.Module):
    def __init__(self, input_dim, m_nodes, s_init=1.0, k_init=1.0):
        super(SinuNetwork, self).__init__()
        # Linear layer: z = Wx + b
        self.linear = nn.Linear(input_dim, m_nodes)
        
        # Learnable parameters s and k (common to all nodes)
        self.s = nn.Parameter(torch.tensor([float(s_init)]))
        self.k = nn.Parameter(torch.tensor([float(k_init)]))

    def forward(self, x):
        # Apply linear transformation
        z = self.linear(x)
        # Apply Sinu CDF Activation
        return SinuCDFFunction.apply(z, self.s, self.k)

# --- Usage Example ---
# Constructing a network: 10 inputs -> 1 hidden layer with 5 nodes
net = SinuNetwork(input_dim=10, m_nodes=5)

# Example training step
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Dummy data
x_train = torch.rand(8, 10) # 8 samples, 10 features
y_target = torch.rand(8, 5)

# Forward and backward
optimizer.zero_grad()
output = net(x_train)
loss = criterion(output, y_target)
loss.backward()
optimizer.step()

print(f"Updated s: {net.s.item():.4f}, Updated k: {net.k.item():.4f}")

RuntimeError: Found dtype Float but expected Double